In [1]:
import pandas as pd
import numpy as np
import networkx as nx
import re
import os
import gc

out_dir = r"C:\Users\user\Downloads\GSE148375_clean"
manifest_path = r"C:\Users\user\Downloads\HumanExome-12-v1-0-B.csv"

def strip_address_suffix(pid):
    return re.sub(r'_\d+$', '', pid)

encoded_df = pd.read_csv(os.path.join(out_dir, "checkpoint7_snp_encoded_012.csv"))
probe_id_array = encoded_df["probe_id"].to_numpy()
sample_cols = encoded_df.columns[1:]
sample_ids = sample_cols.tolist()
X_snp_first = encoded_df[sample_cols].to_numpy(dtype=np.int8)
del encoded_df
gc.collect()

manifest_df = pd.read_csv(manifest_path, skiprows=7, low_memory=False)
non_autosomal = {"X", "Y", "XY", "MT", "0"}
sex_linked_probes = set(manifest_df.loc[manifest_df["Chr"].isin(non_autosomal), "IlmnID"])
sex_linked_core_names = {strip_address_suffix(p) for p in sex_linked_probes}
our_probe_core_names = {strip_address_suffix(p): p for p in probe_id_array}
to_exclude = {our_probe_core_names[c] for c in sex_linked_core_names if c in our_probe_core_names}
del manifest_df
gc.collect()

keep_mask = ~np.isin(probe_id_array, list(to_exclude))
X_auto_int = X_snp_first[keep_mask]
del X_snp_first
gc.collect()

X_f32 = X_auto_int.astype(np.float32)
p_aa = X_f32.mean(axis=1) / 2
denom_aa = np.sqrt(2 * p_aa * (1 - p_aa))
valid_mask = denom_aa > 1e-8
del X_f32
gc.collect()

valid_indices = np.where(valid_mask)[0]
np.random.seed(0)
chosen_idx = np.random.RandomState(0).choice(valid_indices, 5000, replace=False)

X_subset_int = X_auto_int[chosen_idx].astype(np.float64)
del X_auto_int
gc.collect()

p_subset = p_aa[chosen_idx]
denom_subset = denom_aa[chosen_idx]
X_subset_std = ((X_subset_int - 2 * p_subset[:, None]) / denom_subset[:, None]).T
del X_subset_int
gc.collect()

print("Subset shape:", X_subset_std.shape)

sample_corr = np.corrcoef(X_subset_std)
del X_subset_std
gc.collect()
np.fill_diagonal(sample_corr, 0)

print("Correlation matrix rebuilt, shape:", sample_corr.shape)

Subset shape: (3348, 5000)
Correlation matrix rebuilt, shape: (3348, 3348)


In [2]:
import networkx as nx
import pandas as pd

threshold = 0.5
iu = np.triu_indices_from(sample_corr, k=1)
rows, cols = iu
strong = sample_corr[iu] > threshold

pairs = [(sample_ids[rows[k]], sample_ids[cols[k]], sample_corr[iu][k])
         for k in range(len(strong)) if strong[k]]

print(f"Pairs above threshold {threshold}: {len(pairs)}")

G = nx.Graph()
G.add_nodes_from(sample_ids)
for a, b, r in pairs:
    G.add_edge(a, b, weight=r)

clusters = [c for c in nx.connected_components(G) if len(c) > 1]
print(f"Number of related clusters: {len(clusters)}")
print(f"Total samples involved: {sum(len(c) for c in clusters)}")

samples_to_drop = []
for cluster in clusters:
    sorted_cluster = sorted(cluster)
    keep = sorted_cluster[0]
    drop = sorted_cluster[1:]
    samples_to_drop.extend(drop)

print(f"\nSamples to drop: {len(samples_to_drop)}")
print(f"Samples remaining: {len(sample_ids) - len(samples_to_drop)}")

import os
out_dir = r"C:\Users\user\Downloads\GSE148375_clean"
pd.Series(samples_to_drop, name="dropped_sample_id_relatedness").to_csv(
    os.path.join(out_dir, "qc_log_dropped_relatedness.csv"), index=False
)
print("Saved drop log.")

Pairs above threshold 0.5: 375
Number of related clusters: 249
Total samples involved: 561

Samples to drop: 312
Samples remaining: 3036
Saved drop log.


In [3]:
import pandas as pd
import os

out_dir = r"C:\Users\user\Downloads\GSE148375_clean"

samples_to_drop_set = set(samples_to_drop)

meta_df = pd.read_csv(os.path.join(out_dir, "checkpoint2_metadata_sample_filtered.csv"))
meta_df["sample_id"] = meta_df["sample_id"].astype(str)

meta_df_relatedness_filtered = meta_df[~meta_df["sample_id"].isin(samples_to_drop_set)].copy()
print("Metadata shape before:", meta_df.shape)
print("Metadata shape after relatedness filter:", meta_df_relatedness_filtered.shape)

meta_df_relatedness_filtered.to_csv(
    os.path.join(out_dir, "checkpoint2b_metadata_relatedness_filtered.csv"), index=False
)
print("Saved.")

Metadata shape before: (3348, 9)
Metadata shape after relatedness filter: (3036, 9)
Saved.


In [4]:
import pandas as pd
import os

out_dir = r"C:\Users\user\Downloads\GSE148375_clean"
geno_path = os.path.join(out_dir, "checkpoint7_snp_encoded_012.csv")

chunksize = 20000
out_path = os.path.join(out_dir, "checkpoint7b_snp_encoded_012_relatedness_filtered.csv")
first_chunk = True

reader = pd.read_csv(geno_path, chunksize=chunksize)
for chunk in reader:
    cols_to_keep = [c for c in chunk.columns if c == "probe_id" or c not in samples_to_drop_set]
    chunk = chunk[cols_to_keep]
    chunk.to_csv(out_path, mode="w" if first_chunk else "a", header=first_chunk, index=False)
    first_chunk = False

print("Saved:", out_path)
check_df = pd.read_csv(out_path, nrows=0)
print("Sample columns remaining:", len(check_df.columns) - 1, "(expect 3036)")

Saved: C:\Users\user\Downloads\GSE148375_clean\checkpoint7b_snp_encoded_012_relatedness_filtered.csv
Sample columns remaining: 3036 (expect 3036)
